# 将所有的patches转移准备标准化颜色-----记得备份数据

In [ ]:
import os
import shutil
from pathlib import Path
import time

def flatten_dataset():
    # ================= 配置区域 =================
    # 1. 源目录：存放上一元步骤生成的 Patches (里面有很多子文件夹)
    source_root = Path(r"G:\1.Sprouts\High\High.patches.output")
    
    # 2. 目标目录：训练集文件夹 (所有图片将集中到这里)
    train_dir = Path(r"G:\1.Sprouts\High\High.patches.train")
    
    # 3. 操作模式
    # True  = 移动 (剪切)。速度快，不占额外空间，源文件会消失。
    # False = 复制。保留源文件，占用双倍空间。
    MOVE_FILES = True 
    # ===========================================

    # 1. 创建目标目录
    train_dir.mkdir(parents=True, exist_ok=True)

    print(f"📂 源目录: {source_root}")
    print(f"📂 目标目录: {train_dir}")
    print(f"🔧 模式: {'移动 (Move)' if MOVE_FILES else '复制 (Copy)'}")
    print("-" * 50)

    # 2. 扫描所有 PNG 文件
    # rglob('*') 会递归查找所有子文件夹
    png_files = list(source_root.rglob("*.png"))
    
    if not png_files:
        print("❌ 在源目录下没有找到 .png 文件！")
        return

    print(f"🔍 发现 {len(png_files)} 个 Patch 文件，准备开始...")

    count = 0
    start_time = time.time()

    for file_path in png_files:
        # 获取文件名 (例如 TCGA-xxx_ROI1_x100_y100.png)
        file_name = file_path.name
        
        # 构建目标路径
        dest_path = train_dir / file_name

        try:
            if MOVE_FILES:
                # 移动文件
                shutil.move(str(file_path), str(dest_path))
            else:
                # 复制文件
                shutil.copy2(str(file_path), str(dest_path))
            
            count += 1
            if count % 1000 == 0:
                print(f"   已处理 {count} 张...")

        except Exception as e:
            print(f"❌ 处理文件失败 {file_name}: {e}")

    # (可选) 清理空的源文件夹
    if MOVE_FILES:
        print("🧹 正在清理空的源文件夹...")
        for root, dirs, files in os.walk(source_root, topdown=False):
            for name in dirs:
                try:
                    os.rmdir(os.path.join(root, name))
                except OSError:
                    pass # 文件夹非空则忽略

    duration = time.time() - start_time
    print("-" * 50)
    print(f"🎉 转移完成！")
    print(f"📊 共转移: {count} 张图片")
    print(f"⏱️ 耗时: {duration:.2f} 秒")
    print(f"📁 训练集位置: {train_dir}")

if __name__ == "__main__":
    flatten_dataset()
